# 05 - Ensemble Model (Walk-forward + Rolling Window)

This notebook trains and evaluates the stacking ensemble under a **walk-forward, rolling-window** protocol (`window=126`) to match the updated backtesting design.

Pipeline summary:
1. Load base-model predictions for 2024 (validation) and 2025 (test).
2. Run **Phase 1** meta-learning on 2024 using sequential walk-forward updates.
3. Run **Phase 2** forecasting on 2025 with rolling meta-learner refits and one-step-ahead predictions.
4. Compute metrics, generate comparison plots, and export final results.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.models.ensemble import EnsembleModel
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR, MODELS_SAVED_DIR

plotter = Plotter()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

I0000 00:00:1774185611.607907  131169 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774185611.663469  131169 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774185616.013782  131169 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# ── Load data ────────────────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

target_col = 'Close'
y_val  = val[target_col].values
y_test = test[target_col].values

In [3]:
# ── 1. Load Data & Walk-Forward Meta-Learning ───────────────────────────────
import pandas as pd
from src.config import RESULTS_DIR
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.models.ensemble import EnsembleModel

plotter = Plotter()

# 1) Load base-model predictions
df_val_base = pd.read_csv(RESULTS_DIR / 'baseline_val_preds.csv', index_col=0, parse_dates=True)
df_val_lstm = pd.read_csv(RESULTS_DIR / 'lstm_val_preds.csv', index_col=0, parse_dates=True)
df_val_lstm.columns = ['LSTM']

df_test_base = pd.read_csv(RESULTS_DIR / 'baseline_test_preds.csv', index_col=0, parse_dates=True)
df_test_lstm = pd.read_csv(RESULTS_DIR / 'lstm_test_preds.csv', index_col=0, parse_dates=True)
df_test_lstm.columns = ['LSTM']

# 2) Build aligned meta-features
X_meta_train = df_val_base.join(df_val_lstm).dropna()
y_meta_train = val.loc[X_meta_train.index, 'Close']

X_meta_test = df_test_base.join(df_test_lstm).dropna()
y_true_2025 = test.loc[X_meta_test.index, 'Close']

# 3) Walk-forward + rolling-window ensemble
print("--- Training Ensemble with Walk-forward + Rolling Window (126) ---")
ensemble = EnsembleModel(alpha=1.0)

val_pred_dict = {col: X_meta_train[col].values for col in X_meta_train.columns}
test_pred_dict = {col: X_meta_test[col].values for col in X_meta_test.columns}

ens_val_series, ens_test_series = ensemble.train_and_refit_walk_forward(
    val_predictions=val_pred_dict,
    y_val=y_meta_train.values,
    test_predictions=test_pred_dict,
    y_test=y_true_2025.values,
    window=126,
    min_train_size=max(20, len(X_meta_train.columns) * 3),
)

print("\n📊 [Phase 1] 2024 Validation Metrics (Walk-forward Ensemble):")
print(calculate_metrics(y_meta_train, ens_val_series))

print("\n📊 [Phase 2] 2025 Test Metrics (Walk-forward Ensemble):")
print(calculate_metrics(y_true_2025, ens_test_series))

# Optional diagnostic: latest rolling coefficients
try:
    print("\n🔍 Latest Rolling Ensemble Weights:")
    print(ensemble.get_weights().sort_values(ascending=False))
except Exception as exc:
    print(f"Could not fetch rolling weights: {exc}")

# 4) Collect all model predictions for side-by-side comparison
all_pred_dict = {col: X_meta_test[col] for col in X_meta_test.columns}
all_pred_dict['Ensemble'] = ens_test_series

all_metrics = {}
print("\n🏆 ULTIMATE 2025 TEST SET METRICS 🏆")
for name, preds in all_pred_dict.items():
    metrics = calculate_metrics(y_true_2025, preds)
    all_metrics[name] = metrics
    print(f"\n[{name}]")
    print(f"MSE:  {metrics['mse']:.2f} | RMSE: {metrics['rmse']:.2f}")
    print(f"MAE:  {metrics['mae']:.2f} | MAPE: {metrics['mape']:.2f}%")
    print(f"Dir Acc: {metrics['directional_accuracy'] * 100:.2f}%")

# ── 3. Final Visualisations ──────────────────────────────────────────────────

# ── 2024 Validation: 5 models vs actual ─────────────────────────────────────
all_pred_dict_val = {col: X_meta_train[col] for col in X_meta_train.columns}
all_pred_dict_val["Ensemble"] = ens_val_series

plotter.plot_predictions_comparison(
    y_true=y_meta_train,
    predictions=all_pred_dict_val,
    dates=X_meta_train.index,
    title="2024 Validation: All Base Models vs Ensemble",
    filename="ultimate_ensemble_comparison_2024_val.png",
)


plotter.plot_predictions_comparison(
    y_true=y_true_2025,
    predictions=all_pred_dict,
    dates=X_meta_test.index,
    title='2025 Final Showdown: All Base Models vs Ensemble',
    filename='ultimate_ensemble_comparison_2025.png',
)

plotter.plot_metrics_comparison(all_metrics, filename='metrics_bar_chart_2025.png')
print('All final plots generated and saved to reports/figures/ !')

# ── 4. Save the Ultimate Metrics ─────────────────────────────────────────────
final_metrics_df = pd.DataFrame(all_metrics).T
final_metrics_path = RESULTS_DIR / 'ultimate_2025_metrics.csv'
final_metrics_df.to_csv(final_metrics_path)

print(f"\n✅ Fresh metrics successfully saved to {final_metrics_path} !")
print(final_metrics_df)

# ── 5. Save the Ultimate Predictions Data (For Plotting) ───────────────────
final_preds_export = {'Actual_Close': y_true_2025}
final_preds_export.update(all_pred_dict)

final_preds_df = pd.DataFrame(final_preds_export)
final_preds_path = RESULTS_DIR / 'ultimate_2025_predictions.csv'
final_preds_df.to_csv(final_preds_path)

print(f"✅ All 2025 final predictions successfully saved to {final_preds_path} !")
print("\nPreview of first 5 rows:")
print(final_preds_df.head())

--- Training Ensemble with Walk-forward + Rolling Window (126) ---

📊 [Phase 1] 2024 Validation Metrics (Walk-forward Ensemble):
{'mse': 170141.39110575666, 'rmse': 412.48198882588395, 'mae': 321.80459268725264, 'mape': 1.6606758503788333, 'directional_accuracy': 0.5130890052356021}

📊 [Phase 2] 2025 Test Metrics (Walk-forward Ensemble):
{'mse': 651601.4401548003, 'rmse': 807.2183348727904, 'mae': 449.2357729666284, 'mape': 2.052023857732052, 'directional_accuracy': 0.5691489361702128}

🔍 Latest Rolling Ensemble Weights:
xgboost    2.264350e+00
lstm       5.134885e-01
arima     -1.095143e-20
prophet   -1.246365e-01
dtype: float64

🏆 ULTIMATE 2025 TEST SET METRICS 🏆

[ARIMA]
MSE:  8766965.82 | RMSE: 2960.91
MAE:  2623.91 | MAPE: 11.12%
Dir Acc: 1.06%

[Prophet]
MSE:  3123176.73 | RMSE: 1767.25
MAE:  1284.05 | MAPE: 6.13%
Dir Acc: 60.11%

[XGBoost]
MSE:  17765110.81 | RMSE: 4214.87
MAE:  3756.51 | MAPE: 15.61%
Dir Acc: 47.34%

[LSTM]
MSE:  3338389.60 | RMSE: 1827.13
MAE:  1647.93 | MAPE:

## Final Summary

| Metric | Best Model |
|--------|------------|
| RMSE   | See `reports/results/all_models_metrics.csv` |
| MAPE   | See `reports/results/all_models_metrics.csv` |
| Dir. Acc. | See `reports/results/all_models_metrics.csv` |

All results saved under `reports/results/`.